# Markov Chain Model

## Intro

Markov chains can be interesting modelling methods when you have time dependent covariates in your dataset. In their 2014 paper: *Intensity models and transition probabilities for credit card loan delinquencies*, Mindy Leow and Jonathan Crook model credit card delinquency as a non-homogeneous Markov chain with four states. They create a Markov Chain between states of credit. Up to date, 1 month in arrears 2 month in arrears and defaulting. A similar methodology could be applied to our dataset.

## Adaptation

There are three types covariants which could be focused on for this model:

- `PAY_n` (which tracks repayment status month by month)
- `PAY_AMTn` (which tracks amount repayed month by month)
- `BILL_AMTn` (which tracks amount owed month by month)

Pay status is what is looked at in the paper so we will work with the same here. The statuses can be split into 4 brackets, similarly to the paper, and then a transition matrix for going from one state to another can be made using a the training data.

Although the origional paper uses a non-homogeneous markov chain, this model treats the repayment history as a time-homogeneous markov chain. Where probablitiy of moving to any other state is dependent only on the current state. While this is not completely realistic as risk will change with account age, our dataset does not give any information on account age so this simplification is justifiable.

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, accuracy_score, confusion_matrix

In [2]:
# load the cleaned Taiwan data made by 01-Introduction
ROOT = Path.cwd().parent
taiwan = pd.read_csv(ROOT / "data" / "processed" / "taiwan-clean.csv", index_col="ID")

In [3]:
taiwan = taiwan.copy()
taiwan = taiwan.rename(columns={"PAY_0": "PAY_1"})   # rename for consistency with other columns

#order data chronologically by month (PAY_1 is the most recent month, PAY_6 is the oldest)
months = ["Apr", "May", "Jun", "Jul", "Aug", "Sep", "Oct"]
pay_cols  = ["PAY_6", "PAY_5", "PAY_4", "PAY_3", "PAY_2", "PAY_1"]
bill_cols = [f"BILL_AMT{i}" for i in range(6, 0, -1)]
amt_cols  = [f"PAY_AMT{i}" for i in range(6, 0, -1)]

# create table which shows the relevant data from the cleaned dataframe for a given client (row)
def client_table(taiwan, n=0):
    r = taiwan.iloc[n]
    return pd.DataFrame(
        [list(range(1, 8)),
         list(r[pay_cols]) + [r["default payment next month"]],
         list(r[bill_cols]) + [r["default payment next month"]],
         list(r[amt_cols]) + [r["default payment next month"]]],
        index=["Month no.", "Status (PAY / default label)", "BILL_AMT", "PAY_AMT"],
        columns=months,
    )

print(client_table(taiwan, 1))

                               Apr   May   Jun   Jul   Aug   Sep  Oct
Month no.                        1     2     3     4     5     6    7
Status (PAY / default label)     2     0     0     0     2    -1    1
BILL_AMT                      3261  3455  3272  2682  1725  2682    1
PAY_AMT                       2000     0  1000  1000  1000     0    1


## Creating The Transition Matrix

### 1. Splitting Data

The status has values from -2 to 8 Which will be split into categories of lateness:
- If the number is under 0 then they will be grouped together as current.
- One month late
- Two months late 
- Three or more months late

In [4]:
states = ["current", "1_month_late", "2_months_late", "3plus_months_late"]
def to_state(status):
    if status <= 0:
        return "current"
    if status == 1:
        return "1_month_late"
    if status == 2:
        return "2_months_late"
    return "3plus_months_late"

payment_states = taiwan[pay_cols].apply(lambda col: col.map(to_state))

y = taiwan["default payment next month"].astype(int)



### 2. Split Training & Testing

When splitting the data for training and testing the state is fixed so it does the same randomisation each time. `stratify=y` means that the split of default and non-defaulters will be approximately the same in the test set and the training set.

In [5]:

train_idx, test_idx = train_test_split(
    np.arange(len(taiwan)), test_size=0.25, random_state=0, stratify=y
)



### 3. Transition Matrix Estimation

Let $\mathcal{T}$ be the set of training clients. The transition count from state $i$ to state $j$, pooled over clients and over the five consecutive month pairs, is

 $$n_{ij}=\sum_{k\in\mathcal{T}}\ \sum_{t=1}^{5}\mathbf{1}\{S_{k,t}=i,\ S_{k,t+1}=j\}.$$

 This calculates the total number of changes from each state to each other state which can then be used to estimate a probability

 ### 4. Laplace Smoothing

 Add-one (Laplace) smoothing initialises every cell at 1, so

 $$\hat P_{ij}=\frac{n_{ij}+1}{\sum_{l=1}^{4}(n_{il}+1)}=\frac{n_{ij}+1}{n_{i\cdot}+4},\qquad n_{i\cdot}=\sum_l n_{il}.$$

 This is the posterior mean under a symmetric Dirichlet $(1,1,1,1)$ prior on each row, and it guarantees $\hat P_{ij}>0$ (no zero-probability transitions). This is important because the "true probability" of an event while the data may not include an instance is likely not zero and having a small boost will not significantly affect large datasets.

In [6]:
#estimate the transition matrix 
transition_counts = pd.DataFrame(1.0, index=states, columns=states)  #laplace smoothing
for idx in train_idx:
    sequence = payment_states.iloc[idx].tolist()
    for source, destination in zip(sequence[:-1], sequence[1:]):
        transition_counts.loc[source, destination] += 1

transition_matrix = transition_counts.div(transition_counts.sum(axis=1), axis=0)

### 5. State-Level default rate

Let $N_s= \#\{k\in\mathcal{T}: S_{k,6}=s\}$ and $D_s=\sum_{k\in\mathcal{T}}Y_k\,\mathbf{1}\{S_{k,6}=s\}$, where $S_{k,6}$ is the latest state (`PAY_1`). The global training default rate is

 $$\bar r=\frac{1}{|\mathcal{T}|}\sum_{k\in\mathcal{T}}Y_k.$$

 The smoothed default rate for state $s$ uses $m=20$ pseudo-observations:

 $$\hat r_s=\frac{D_s+m\,\bar r}{N_s+m}=\underbrace{\frac{N_s}{N_s+m}}_{w_s}\frac{D_s}{N_s}+\bigl(1-w_s\bigr)\bar r.$$

 This is a weighted average of the raw state rate $D_s/N_s$ and the global rate $\bar r$. The weight $w_s\to 1$ for common states and $w_s\to 0$ for rare ones. This will increase the bias but also reduce noise and variance in the data

In [7]:
#smoothed default rate for each observed payment state
train_states = payment_states.iloc[train_idx]["PAY_1"]
global_default_rate = y.iloc[train_idx].mean()
state_default_rate = {}

for state in states:
    mask = train_states.eq(state)
    state_default_rate[state] = (
        y.iloc[train_idx][mask].sum() + 20 * global_default_rate
    ) / (mask.sum() + 20)



### 6. Prediction 
For a test client $i$ whose latest state is $s_i=S_{i,6}$, we project one transition forward and take the expected state-level default rate:

 $$\hat p_i=\sum_{j=1}^{4}\hat P_{s_i j}\,\hat r_j=\mathbf{e}_{s_i}^{\top}\hat P\,\hat{\mathbf r},$$

 where $\mathbf{e}_{s_i}$ is the unit vector for state $s_i$ and $\hat{\mathbf r}=(\hat r_1,\dots,\hat r_4)^\top$. More generally, for a horizon of $h$ steps,

 $$\hat p_i^{(h)}=\mathbf{e}_{s_i}^{\top}\hat P^{\,h}\hat{\mathbf r}.$$


In [8]:
#project one Markov transition from each client's latest payment state.
test_current_states = payment_states.iloc[test_idx]["PAY_1"]
risk_by_state = pd.Series(state_default_rate)
predicted_default_probability = (
    transition_matrix.loc[test_current_states].to_numpy() @ risk_by_state[states].to_numpy())

y_test = y.iloc[test_idx].to_numpy()


### 7. Performance Metrics

With a threshold $\tau$ (here $\tau=0.5$) the predicted class is $\hat Y_i=\mathbf{1}\{\hat p_i\ge\tau\}$. With confusion-matrix counts $TP,FP,TN,FN$ on the test set:

 $$\text{Accuracy}=\frac{TP+TN}{TP+TN+FP+FN},$$

 $$\text{Precision}=\frac{TP}{TP+FP},\qquad \text{Recall}=\frac{TP}{TP+FN}.$$

 The ROC AUC is the probability that a randomly chosen defaulter is scored higher than a randomly chosen non-defaulter:

 $$\text{AUC}=\mathbb{P}\bigl(\hat p_{i}>\hat p_{k}\mid Y_i=1,\,Y_k=0\bigr)+\tfrac12\,\mathbb{P}\bigl(\hat p_{i}=\hat p_{k}\mid Y_i=1,\,Y_k=0\bigr).$$

In [9]:
print("Transition matrix:")
display(transition_matrix)
print(f"ROC AUC: {roc_auc_score(y_test, predicted_default_probability):.3f}")
print(f"Accuracy (0.5 threshold): {accuracy_score(y_test, predicted_default_probability >= 0.5):.3f}")
print("Confusion matrix:")
print(confusion_matrix(y_test, predicted_default_probability >= 0.5))

Transition matrix:


,current,1_month_late,2_months_late,3plus_months_late
current,0.938390,0.014174,0.047426,0.000010
1_month_late,0.035714,0.892857,0.035714,0.035714
2_months_late,0.249716,0.102404,0.581533,0.066347
3plus_months_late,0.107438,0.086088,0.287879,0.518595


ROC AUC: 0.694
Accuracy (0.5 threshold): 0.823
Confusion matrix:
[[5624  216]
 [1108  551]]


## Improvements

This model returns an accuracy only slightly higher than the baseline of predicting everyone wont default and has a large proportion of false negatives. This is an important metric when the goal is to identify defaulters in advance. 

Some things which could be further looked at with this model to try and improve its performace are:

- Experiment with the other covariants and compare the metrics
- Add calibration step
- Tune the smoothing using cross validation
- Adjust how the `PAY_n` data is split into bins 
- Adjust decision threshold to balance the cost of false postivies and negatives 
- Modelling the chain as non-homogeneous

#### References

- Leow, M. and Crook, J. (2014). Intensity models and transition probabilities for credit card loan delinquencies. *European Journal of Operational Research*, 236(2), pp.685–694. doi:10.1016/j.ejor.2013.12.026. Available at: https://www.research.ed.ac.uk/en/publications/intensity-models-and-transition-probabilities-for-credit-card-loa-5/ [Accessed 6 Oct. 2026].
- Hoff, P. (2025). Shrinkage and empirical Bayes. [online] Available at: https://www2.stat.duke.edu/~pdh10/Teaching/732/Notes/shrinkage.pdf [Accessed 6 Oct. 2026].
- Chen, S.F. and Goodman, J. (1999). An empirical study of smoothing techniques for language modeling. Computer Speech & Language, [online] 13(4), pp.359–394. doi:10.1006/csla.1999.0128.